# Tutorial 1
# Build and train your own neural network from scratch
By: Dr Tiffany Vlaar (University of Glasgow)

***
### Load in necessary packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

***
## Neural Network
Let us create a small neural network with one hidden layer.

In [13]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.net = nn.Sequential(nn.Linear(2,2),               # Input to hidden layer
                                 nn.Tanh(),                    # tanh activation
                                 nn.Linear(2,1, bias = False), # Hidden layer to output, set bias to False
                                 )

    def forward(self, x):
        out = self.net(x)
        return out

# Initialize the neural network
NN = Net()

*Task:* What are the dimensions of the neural network parameters?  



# Example data

In [14]:
x = torch.tensor([[1.0, 2.0]]) # input
y = torch.tensor([[1.0]])      # true label

# Feed through neural network
output = NN(x)

# Define the loss, here given as an example is the mean square error loss
criterion = nn.MSELoss()

# Compute the loss between the neural network's output and true label y
loss = criterion(output,y)

# Print the loss
print(loss)

tensor(1.6650, grad_fn=<MseLossBackward0>)


*Task:* Write your own function to compute the mean square error loss, and verify you obtain the same loss as above.

**Backpropagation**

In [15]:
# Backpropagation
NN.zero_grad()
loss.backward()

# Print the gradients
for name, p in NN.named_parameters():
    print(name)
    print(p)
    print("gradients",p.grad)
    print(" ")

net.0.weight
Parameter containing:
tensor([[-0.0221,  0.1685],
        [-0.4552,  0.4380]], requires_grad=True)
gradients tensor([[ 0.7407,  1.4814],
        [-0.3251, -0.6502]])
 
net.0.bias
Parameter containing:
tensor([ 0.3623, -0.6571], requires_grad=True)
gradients tensor([ 0.7407, -0.3251])
 
net.2.weight
Parameter containing:
tensor([[-0.4400,  0.1331]], requires_grad=True)
gradients tensor([[-1.5218,  0.5986]])
 


*Task:* Write down the gradients of the loss with respect to the parameters and verify that your derivation was correct by comparing it with the gradients obtained above.

*Hint:* You can make this easier for yourself by explicitly setting the initial neural network parameters yourself (instead of using NN = Net()), for example using the commands below:

```
with torch.no_grad():
    NN.net[0].weight[:] = torch.tensor([
        [0.1, 0.2],
        [-0.1, 0.1]
    ])

    NN.net[0].bias[:] = torch.tensor([
         0.0,
         0.0
    ])

    NN.net[2].weight[:] = torch.tensor([
        [0.3, -0.2]
    ])

```



**Parameter update using gradient descent**

*Task:* Write out the gradient descent algorithm on a piece of paper.  Apply gradient descent with a specific step size of your choice using the gradients you computed above. What are the updated parameter values?

You can use the code below to check your answer.

In [5]:
NN = Net() #initialise network
output = NN(x)  # feed through neural network
loss = criterion(output,y) # compute the loss
print("loss before update", loss.item())

NN.zero_grad()
loss.backward()

#define the step size
h = 0.1

# Update the parameters using gradient descent
for name, p in NN.named_parameters():
    with torch.no_grad():
        p -= h*p.grad

    #print the updated parameter values
    print(name)
    print("updated p", p)
    print(" ")

# compute the loss after the update
output = NN(x)
loss = criterion(output,y)
print("loss after update", loss.item())

loss before update 1.3398195505142212
net.0.weight
updated p Parameter containing:
tensor([[-0.1954,  0.1375],
        [-0.0662, -0.4296]], requires_grad=True)
 
net.0.bias
updated p Parameter containing:
tensor([0.6990, 0.6938], requires_grad=True)
 
net.2.weight
updated p Parameter containing:
tensor([[0.5180, 0.2545]], requires_grad=True)
 
loss after update 0.5186904668807983


## From Scratch

In [6]:
initlayer1 = nn.Linear(2,5)
initlayer2 = nn.Linear(5,2,bias=False)
W1 = initlayer1.weight
W2 = initlayer2.weight
B1 = initlayer1.bias


*Task:* What are the dimensions of the neural network parameters?  


In [7]:
def derivative_tanh(x):
    return 1/torch.cosh(x)**2

class myNet():
    def __init__(self,W1,W2,B1):
        self.W1 = W1
        self.B1 = B1
        self.W2 = W2

    def forward(self,x):
        self.x = x
        self.fw1 = self.x@self.W1.T+self.B1
        self.layer1 = torch.tanh(self.fw1)
        self.layer2 = self.layer1@self.W2.T
        return self.layer2

    def backward(self,y):
        self.y = y
        grad = -(1/self.y.size()[0])*(self.y-self.layer2)
        self.dL_dW2 = grad.T@self.layer1

        grad = (grad@self.W2)*derivative_tanh(self.fw1)
        self.dL_dW1 = grad.T@self.x
        self.dL_dB1 = torch.sum(grad,axis=0)

    def train(self,h):
        self.W1.data -= h*self.dL_dW1
        self.W2.data -= h*self.dL_dW2
        self.B1.data -= h*self.dL_dB1

In [8]:
#XOR problem
inputs = torch.FloatTensor([[0,0],[0,1],[1,0],[1,1]])
targets = torch.FloatTensor([[1,0],[0,1],[0,1],[1,0]])

In [9]:
# At initialization
net = myNet(W1,W2,B1)
outs = net.forward(inputs)
print(outs)

tensor([[-0.0618,  0.0646],
        [-0.2097,  0.0389],
        [-0.2297,  0.0861],
        [-0.3635,  0.0941]], grad_fn=<MmBackward0>)


In [10]:
# Update parameters
for i in range(500):
    outs = net.forward(inputs)
    net.backward(targets)
    net.train(h=0.2)
    if i % 20 == 0: # print the loss after every 20 parameter updates
        print("loss",torch.sum((outs-targets)**2).detach().numpy())

# print the final output
print(" ")
outs = net.forward(inputs).detach()
print("net output \n",outs.numpy())
print(" ")
print("target \n",targets.numpy())


loss 4.8550744
loss 2.0164382
loss 1.9705372
loss 1.9301976
loss 1.8744576
loss 1.7831087
loss 1.628236
loss 1.3773222
loss 1.0225046
loss 0.63145614
loss 0.32079232
loss 0.13995457
loss 0.05591066
loss 0.021655371
loss 0.008472895
loss 0.0034303297
loss 0.001450011
loss 0.0006386544
loss 0.00029085536
loss 0.00013574614
loss 6.442202e-05
loss 3.0907613e-05
loss 1.49288835e-05
loss 7.242472e-06
loss 3.522054e-06
 
net output 
 [[ 1.0002259e+00  2.5837123e-04]
 [-4.5236945e-04  9.9943280e-01]
 [-4.2094290e-04  9.9944454e-01]
 [ 1.0004580e+00  6.1339140e-04]]
 
target 
 [[1. 0.]
 [0. 1.]
 [0. 1.]
 [1. 0.]]


**Challenge:** modify the above to include a bias in the output layer.

**Challenge**: Modify the below to use a sigmoid activation function in the output layer and a binary cross entropy (BCE) loss. Update the forward and backward passes accordingly.

In [17]:
#XOR problem
inputs = torch.FloatTensor([[0,0],[0,1],[1,0],[1,1]])
targets = torch.FloatTensor([[0],[1],[1],[0]])

initlayer1 = nn.Linear(2,5)
initlayer2 = nn.Linear(5,1,bias=False)
W1 = initlayer1.weight
W2 = initlayer2.weight
B1 = initlayer1.bias


#### your code goes here
#def sigmoid(x):
#      return


class myNet():
    def __init__(self,W1,W2,B1):
        self.W1 = W1
        self.B1 = B1
        self.W2 = W2

    def forward(self,x):
        ##### your code goes here
        self.x = x
        #return

    def backward(self,y):
        ##### your code goes here
        self.y = y


    def train(self,h):
        self.W1.data -= h*self.dL_dW1
        self.W2.data -= h*self.dL_dW2
        self.B1.data -= h*self.dL_dB1

net = myNet(W1,W2,B1)
outs = net.forward(inputs)

# Update parameters
for i in range(500):
    outs = net.forward(inputs)
    net.backward(targets)
    net.train(h=0.2)
    if i % 20 == 0: # print the loss after every 20 parameter updates
        print("loss",torch.sum((outs-targets)**2).detach().numpy())

# print the final output
print(" ")
outs = net.forward(inputs).detach()
print("net output \n",outs.numpy())
print(" ")
print("target \n",targets.numpy())

AttributeError: 'myNet' object has no attribute 'dL_dW1'